In [ ]:
import numpy as np
import h5py
import hdf5plugin
import matplotlib.pyplot as plt
from matplotlib import colors
from pathlib import Path
from collections import OrderedDict
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.animation import FuncAnimation
from collections import OrderedDict
from IPython.display import HTML,display

In [ ]:
def animate_image_subsets(data_dict, key_ranges=None, title_dict=None, interval=200, save_path=None):
    """
    Generates and displays separate animations for each specified range of keys.
    
    Parameters:
    -----------
    data_dict : dict or OrderedDict
        Your dictionary containing {float_key: image_array}
    key_ranges : list of tuples
        A list of inclusive (min, max) bounds. Each tuple creates its own animation.
        Example: [(126, 132), (136, 147)]
    title_dict : dict, optional
        Your dictionary containing {float_key: title_string}. If None, defaults to the float key.
    interval : int, default 200
        Time between frames in milliseconds.
    save_path : str, optional
        Base filename for saving. The range bounds will be automatically appended.
    """
    animation_widgets = []
    if not key_ranges:
        key_ranges = [(min(data_dict.keys()), max(data_dict.keys()))]
    for start, end in key_ranges:
        # 1. Filter items belonging strictly to this specific subset range
        subset_items = [
            (k, v) for k, v in data_dict.items() 
            if start <= k <= end
        ]
        # Explicitly sort the subset by its float keys
        subset_items.sort(key=lambda item: item[0])
        
        if not subset_items:
            print(f"Skipping range ({start} to {end}): No matching keys found.")
            continue
            
        keys = [item[0] for item in subset_items]
        frames = [item[1] for item in subset_items]
        
        # 2. Initialize a fresh figure and axis for this specific animation sequence
        fig, ax = plt.subplots()
        ax.axis('off')
        
        im = ax.imshow(frames[0])
        
        # Determine the initial title string based on whether title_dict was provided
        initial_title = title_dict[keys[0]] if title_dict else f"Key: {keys[0]:.4f}"
        ax.set_title(initial_title)
        
        # 3. Define the frame update logic 
        def update(frame_index, keys=keys, frames=frames, im=im, ax=ax):
            im.set_data(frames[frame_index])
            current_key = keys[frame_index]
            
            # Determine the title string dynamically for the current frame
            frame_title = title_dict[current_key] if title_dict else f"Key: {current_key:.4f}"
            ax.set_title(frame_title)
            return [im]
            
        # 4. Construct the unique animation object
        ani = FuncAnimation(
            fig, update, frames=len(frames), interval=interval, blit=True
        )
        
        # 5. Handle automatic file renaming if saving is requested
        if save_path:
            path = Path(save_path)
            start_str = str(start).replace('.', '_')
            end_str = str(end).replace('.', '_')
            
            # Construct new filename structure: parent_dir / "filename_start_to_end.ext"
            subset_path = path.parent / f"{path.stem}_{start_str}_to_{end_str}{path.suffix}"
            
            fps = int(1000 / interval)
            file_extension = subset_path.suffix.lower()
            
            if file_extension == '.gif':
                print(f"Saving range [{start}, {end}] to {subset_path.name}...")
                ani.save(subset_path, writer='pillow', fps=fps)
            elif file_extension == '.mp4':
                print(f"Saving range [{start}, {end}] to {subset_path.name}...")
                ani.save(subset_path, writer='ffmpeg', fps=fps, bitrate=1800)
            else:
                print(f"Warning: Extension '{file_extension}' not recognized. Save skipped.")
        
        # 6. Close the background canvas to prevent duplicate static image renders
        plt.close(fig)
        
        # 7. Generate HTML player widget, print a header, and push it live into the notebook
        html_player = HTML(ani.to_jshtml())
        print(f"\n=== Animation Range: {start} to {end} ===")
        display(html_player)
        
        animation_widgets.append(html_player)
        
    return animation_widgets

In [ ]:
# single gate measurement
# need to adjust the location and pattern of the h5 files
h5_loc=Path(r'X:\bl17-2\2026_06_Lindenberg\BTO_001') #valid path from beamline analysis computer during June
h5_pattern = 'BTO_001_extg_singlemode_manual-settings-test2_peak002_fcngen_w30ns_delay*data*.h5'
h5_files=h5_loc.glob(h5_pattern)
intensities=OrderedDict()
for file in h5_files:
    print(file.stem.split('_')[-3])
    delay=file.stem.split('_')[-3].split('delay')[-1].split('ns')[0]
    data=h5py.File(file,'r')['entry/data/data'][0,300:450,300:700]
    data=np.where(data>4e8,np.nan, data)
    intensity=np.nansum(data)
    intensities[delay] = intensity
    print(f'File: {file.stem}, Delay: {delay} ns, Intensity: {intensity}')
    fig,ax=plt.subplots()
    im=ax.imshow(data,norm=colors.LogNorm())
    ax.set_title(f'Delay: {delay} ns')

In [ ]:
# test_file=h5py.File(list(h5_files)[0],'r')
# test_file.visititems(print)
# fig,ax=plt.subplots()
# ax.imshow(test_file['entry/data/data'][0],norm=colors.LogNorm())

In [ ]:
def plot_intensities(intensities):
    delays = np.array(list(intensities.keys()), dtype=float)
    print(delays)
    intensity_values = np.array(list(intensities.values()),dtype=float)
    #intensity_values = intensity_values / np.mean(intensity_values)  # Normalize to max value
    
    plt.figure(figsize=(10, 6))
    plt.plot(delays, intensity_values, marker='o',linestyle='',color='blue')
    plt.title('Intensity vs Delay')
    plt.xlabel('Delay (ns)')
    plt.ylabel('Intensity (sum of pixel values)')
    plt.grid()
    plt.show()

In [ ]:
plot_intensities(intensities)

In [ ]:
# dual gate measurement
# need to adjust the location and pattern of the h5 files
h5_loc=Path(r'X:\bl17-2\2026_06_Lindenberg\BTO_001')
h5_pattern = 'BTO_001_extg_dualmode_peak002_fcngen_w30ns_diff163ns_delay*_6_data*.h5'
h5_files=h5_loc.glob(h5_pattern)
intensities_on=OrderedDict()
intensities_off=OrderedDict()
for file in h5_files:
    print(file.stem.split('_')[-4])
    delay=file.stem.split('_')[-4].split('delay')[-1].split('ns')[0]
    data_on=h5py.File(file,'r')['entry/data/data'][0,300:450,300:700]
    data_off=h5py.File(file,'r')['entry/data/data'][1,300:450,300:700]
    data_on=np.where(data_1>4e8,np.nan, data_1)
    data_off=np.where(data_off>4e8,np.nan, data_off)
    intensity_on=np.nansum(data_1)
    intensity_off=np.nansum(data_off)
    intensities_on[delay] = intensity_on
    intensities_off[delay] = intensity_off
    print(f'File: {file.stem}, Delay: {delay} ns, Intensity On: {intensity_on}, Intensity Off: {intensity_off}')
    fig,ax=plt.subplots(1,3)
    im=ax[0].imshow(data_1,norm=colors.LogNorm(vmin=1e3,vmax=1e10))
    im=ax[1].imshow(data_off,norm=colors.LogNorm(vmin=1e3,vmax=1e10))
    plt.colorbar(im, ax=ax[1])
    fig.suptitle(f'Delay: {delay} ns')

In [ ]:
plot_intensities(intensities_on)
plot_intensities(intensities_off)